# 08 - Objedinjeni zaključak projekta

Ova sveska je završna sinteza celog projekta *Telco Customer Churn*.
Prethodne sveske (`00`-`07`) su svaka pokrivale po jedan korak analize sa
svojim, lokalnim zaključkom, a ovde sve te nalaze povezujemo u
jedinstvenu celinu: šta smo saznali o korisnicima koji napuštaju
kompaniju, koji je model najbolji izbor i zašto, i šta bi kompanija
konkretno trebalo da uradi na osnovu svega ovoga.

## 1. Tok projekta: pregled

| Sveska | Sadržaj |
|---|---|
| `00` | Definisanje problema, konteksta i cilja (binarna klasifikacija) |
| `01` | Inicijalni pregled: dimenzije, tipovi, skrivene nedostajuće vrednosti u `TotalCharges` |
| `02` | Provera sistematskih/logičkih grešaka između povezanih kolona |
| `03` | Stratifikovana podela na trening (80%) i test (20%) skup |
| `04` | Detekcija statističkih anomalija (IQR), skup je statistički čist |
| `05` | Feature engineering (`ActiveServices`, `TenureGroup`) i enkodiranje |
| `06` | Iterativna EDA, statistička potvrda ključnih prediktora |
| `07` | Modelovanje, cross-validacija, tuning hiperparametara, ROC krive |
| `08` | *(ova sveska)*, sinteza i poslovne preporuke |

Kroz ceo proces smo se držali istog principa: sve odluke o čišćenju,
transformaciji i feature engineering-u donosimo isključivo na trening
skupu, dok test skup ostaje netaknut sve do finalne evaluacije. Tako smo
izbegli *data leakage*.

## 2. Ključni nalazi o korisnicima koji napuštaju kompaniju

Iz statističke analize (sveska `06`) i analize važnosti atributa (sveska
`07`) izdvaja se nekoliko obrazaca, svi statistički potvrđeni (p < 0.05)
i prisutni kod sva tri modela.

Tip ugovora se pokazao kao ubedljivo najjači pojedinačni faktor.
Korisnici sa `Month-to-month` ugovorom napuštaju kompaniju u oko 43%
slučajeva, naspram svega oko 3% kod dvogodišnjih ugovora. Kod XGBoost-a
`Contract` nosi čak oko 30% ukupne važnosti među svim atributima, daleko
ispred svega ostalog.

Dužina pretplate (`tenure`) je takođe usko povezana sa odlaskom.
Korisnici koji odlaze imaju u proseku mnogo kraći staž, što ima smisla,
rizik je najveći u prvim mesecima, pre nego što se korisnik "navikne" na
uslugu.

Tip internet usluge igra veću ulogu nego što bismo očekivali. Korisnici
sa `Fiber optic` internetom imaju, paradoksalno, viši churn (oko 42%) od
korisnika sa `DSL` (oko 19%). Mogući razlog je viša cena ili razlike u
kvalitetu usluge i konkurenciji na tom delu tržišta, ovo je nešto što bi
kompanija trebalo dodatno da istraži.

Način plaćanja takođe govori nešto o angažovanosti korisnika. Korisnici
koji plaćaju preko `Electronic check`-a, najmanje automatizovanog načina
plaćanja, napuštaju znatno češće (oko 45%) od onih sa automatskim
plaćanjem (oko 15-19%).

Nedostatak dodatnih usluga, posebno `TechSupport` i `OnlineSecurity`,
takođe povećava rizik od odlaska. Ovo je bila i motivacija za novi
feature `ActiveServices`, koji se pokazao kao koristan prediktor kod
modela baziranih na stablima.

Treba napomenuti da su ovo sve korelacije potvrđene testovima i
modelima, ne dokazana uzročno-posledična veza. Ne znamo, na primer, da
li duži ugovor zaista uzrokuje lojalnost, ili jednostavno lojalniji
korisnici biraju duže ugovore. Da bismo to zaista utvrdili, trebalo bi
sprovesti kontrolisan eksperiment, na primer A/B test ponude dužih
ugovora.

In [1]:
import pandas as pd

# Rezime finalnih rezultata sva tri modela (pre podešavanja hiperparametara)
rezime_finalni = pd.DataFrame({
    "Model": ["Logistička regresija", "Random Forest", "XGBoost"],
    "Accuracy": [0.7242, 0.7839, 0.7377],
    "Precision": [0.4884, 0.6250, 0.5043],
    "Recall": [0.7888, 0.4679, 0.7754],
    "F1-score": [0.6033, 0.5352, 0.6112],
    "ROC-AUC": [0.8342, 0.8214, 0.8314]
})
print("Rezultati na test skupu (originalni modeli):")
display(rezime_finalni)

rezime_tuning = pd.DataFrame({
    "Model": ["Logistička regresija (posle tuning-a)", "XGBoost (posle tuning-a)"],
    "Accuracy": [0.7249, 0.7306],
    "Precision": [0.4892, 0.4959],
    "Recall": [0.7861, 0.8021],
    "F1-score": [0.6031, 0.6129],
    "ROC-AUC": [0.8347, 0.8417]
})
print("\nRezultati posle podešavanja hiperparametara:")
display(rezime_tuning)

Rezultati na test skupu (originalni modeli):


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistička regresija,0.7242,0.4884,0.7888,0.6033,0.8342
1,Random Forest,0.7839,0.6250,0.4679,0.5352,0.8214
2,XGBoost,0.7377,0.5043,0.7754,0.6112,0.8314



Rezultati posle podešavanja hiperparametara:


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistička regresija (posle tuning-a),0.7249,0.4892,0.7861,0.6031,0.8347
1,XGBoost (posle tuning-a),0.7306,0.4959,0.8021,0.6129,0.8417


## 3. Koji model preporučujemo: finalna odluka

Nijedan model nije bolji od ostalih po svim metrikama, pa izbor zavisi od
toga šta nam je poslovno prioritet.

Random Forest ima najveću Accuracy (78,4%) i Precision (62,5%), ali i
najniži Recall (46,8%), propušta više od polovine korisnika koji zaista
odlaze. Cross-validacija iz sveske `07` je pokazala da je ovaj rezultat
čak i malo optimističan, prosečan CV recall je svega oko 49,8%.

Logistička regresija ima najviši Recall (78,9%) i ROC-AUC (83,4%), i uz
to je najjednostavnija i najlakša za tumačenje, koeficijenti se direktno
čitaju.

XGBoost je najbolje balansiran (najviši F1-score), a posle podešavanja
hiperparametara dostiže i najviši ROC-AUC (84,2%), uz Recall od 80,2%,
što je ukupno najbolji rezultat od svih koje smo testirali.

Kako je kod predviđanja churn-a mnogo skuplje propustiti korisnika koji
odlazi (False Negative) nego pogrešno uzbuniti se oko lojalnog korisnika
(False Positive), Recall nam je ovde prioritet. Zbog toga preporučujemo
XGBoost sa podešenim hiperparametrima kao finalni model, ima najbolju
kombinaciju Recall-a (80,2%) i ROC-AUC-a (84,2%) od svih varijanti koje
smo probali. Logistička regresija ostaje dobra alternativa tamo gde je
bitnija interpretabilnost, na primer kad treba objasniti odluku
korisničkoj podršci ili regulatoru.

## 4. Vrednost feature engineering-a

Nove promenljive koje smo napravili u svesci `05` su se pokazale
korisnim.

`ActiveServices` kod Random Forest-a i XGBoost-a ima umeren do visok
uticaj, što znači da agregiranje šest kolona dodatnih usluga u jedan
broj nije bila samo kozmetička promena, već stvarno korisna
transformacija.

`TenureGroup` ima manji, ali i dalje prisutan značaj kod modela
baziranih na stablima (oko 5-6% važnosti kod oba). Ovi modeli i sami
umeju da pronađu slične pragove direktno iz sirovog `tenure`, ali
grupisana verzija ipak pomaže linearnom modelu, Logističkoj regresiji, da
uhvati nelinearne efekte.

## 5. Ograničenja rada i predlozi za dalje unapređenje

Vredi iskreno reći i gde ovaj rad ima granice, to je i samo po sebi deo
dobre analitičke prakse.

Performanse su solidne, ali ne izuzetne. F1-score od oko 0,61 nije loš
rezultat za skup ove veličine i ovakve nebalansiranosti, ali svakako ima
prostora za bolje. Kao sledeći korak, moglo bi se probati SMOTE za
sintetičko balansiranje klasa umesto samo `class_weight`, ili šira
pretraga hiperparametara, na primer veći `n_iter` kod
RandomizedSearchCV-a.

Takođe, kao što smo već pomenuli u sekciji 2, ono što smo pronašli su
korelacije, a ne dokazana uzročnost. Za poslovne odluke većeg obima bilo
bi pametno te nalaze dodatno proveriti kroz kontrolisan eksperiment.

Tu je i pitanje multikolinearnosti: `tenure` i `TotalCharges` su jako
korelisani (sveska `06`), zbog čega njihov pojedinačni značaj kod
Logističke regresije deluje "razvodnjeno". Model bi verovatno mogao da se
pojednostavi izbacivanjem jedne od te dve kolone.

Na kraju, ovaj rad se zaustavlja na evaluaciji na statičnom test skupu i
ne prati kako bi se model ponašao u produkciji. Za realnu primenu bi
trebalo periodično ponovo trenirati model i pratiti da li mu performanse
padaju tokom vremena, kako se ponašanje korisnika menja.

## 6. Poslovne preporuke

Na osnovu svega što smo pronašli, evo konkretnih preporuka za
telekomunikacionu kompaniju.

Prvo, vredi prioritetno targetirati korisnike sa Month-to-month
ugovorom, to je najveći pojedinačni faktor rizika, i ponuda popusta za
prelazak na godišnji ili dvogodišnji ugovor bi mogla značajno da smanji
odliv.

Drugo, posebnu pažnju treba posvetiti prvih 6-12 meseci pretplate, jer je
to period kad je rizik od odlaska najveći, pa bi tu imalo smisla uvesti
proaktivnu podršku ili uvodne popuste.

Treće, korisnike bi trebalo podsticati da koriste dodatne usluge poput
`TechSupport` i `OnlineSecurity`, jer su ti korisnici lojalniji,
verovatno zato što bi prelaskom kod drugog provajdera izgubili više
funkcionalnosti.

Četvrto, vredelo bi dodatno istražiti zašto je churn kod korisnika sa
Fiber optic internetom toliki, nije jasno da li je razlog cena, kvalitet
usluge ili konkurencija, i pre donošenja konkretnih mera trebalo bi to
dodatno analizirati.

Peto, korisnike bi trebalo podsticati da pređu na automatsko plaćanje,
s obzirom na to da korisnici koji plaćaju preko `Electronic check`-a
odlaze znatno češće, pa bi olakšavanje prelaska na plaćanje karticom ili
preko banke moglo pomoći.

## 7. Zaključna reč

Kroz ovaj projekat smo prošli ceo tok analize podataka, od sirovog CSV
fajla, preko čišćenja i statistički potvrđene eksplorativne analize,
feature engineering-a, pa sve do tri istrenirana i validirana modela sa
sistematski podešenim hiperparametrima. Rezultati pokazuju da je churn
kod ovog skupa podataka predvidiv u razumnoj meri (ROC-AUC oko 0,84), a
faktori rizika koje smo identifikovali imaju smisla i sa poslovne, ne
samo statističke strane.

Kod, dataset i sve sveske dostupni su u GitHub repozitorijumu tima.